# Frequency

## Setup

In [2]:
import sys
import torch
from pathlib import Path

# Local: notebook is in notebooks/, project root is one level up.
if Path('/content/data').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

from transformer_lens import HookedTransformer
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/trishasalas/Repos/Research/tmlr


In [3]:
# Cell 1: Device check
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [4]:
model_name = "EleutherAI/pythia-160m"

model = HookedTransformer.from_pretrained(model_name)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-160m into HookedTransformer
Layers: 12
Heads: 12
Hidden size: 768
Params: 162.3M


In [5]:
from src.frequency import (
        COMPOUNDS, load_trajectories
    )

In [8]:
from src.frequency import (
        query_infinigram, build_frequency_table
    )
# Get corpus frequencies
freq_df = build_frequency_table()
freq_df

Querying: screen_reader...
Querying: alt_text...
Querying: skip_link...
Querying: color_contrast...
Querying: keyboard_navigation...
Querying: focus_indicator...
Querying: semantic_html...
Querying: captions...
Querying: WCAG...
Querying: ARIA...


,compound,word1,word2,bigram_count,word1_count,word2_count,conditional_prob
0,screen_reader,screen,reader,32100.0,22952059,6113770.0,0.001399
1,alt_text,alt,text,23306.0,5902287,31625337.0,0.003949
2,skip_link,skip,link,662.0,2340742,18277281.0,0.000283
3,color_contrast,color,contrast,20112.0,25414097,16674541.0,0.000791
4,keyboard_navigation,keyboard,navigation,8050.0,2791750,2976535.0,0.002883
5,focus_indicator,focus,indicator,1261.0,20169251,2923114.0,0.000063
6,semantic_html,semantic,HTML,2595.0,1358029,4327636.0,0.001911
7,captions,closed,captions,7942.0,16202434,214709.0,0.000490
8,WCAG,WCAG,NaN,NaN,15389,NaN,NaN
9,ARIA,ARIA,NaN,NaN,85321,NaN,NaN


In [9]:
from src.frequency import (
        token_competition_trace
    )
# Trace token competition at a specific scale
comp = token_competition_trace(model, "A skip link is", top_k=10)
comp
traj_df = load_trajectories(PROJECT_ROOT)
traj_df


,suite,concept,compound,trajectory
0,pythia,ARIA,ARIA,never_emerges
1,pythia,WCAG,WCAG,monotonic_climb
2,pythia,alt text,alt_text,monotonic_climb
3,pythia,captions,captions,never_emerges
4,pythia,color contrast,color_contrast,mixed
5,pythia,focus indicator,focus_indicator,never_emerges
6,pythia,keyboard navigation,keyboard_navigation,peak_regress
7,pythia,screen reader,screen_reader,monotonic_climb
8,pythia,semantic HTML,semantic_html,never_emerges
9,pythia,skip link,skip_link,peak_regress


In [10]:
from src.frequency import (
        frequency_trajectory_correlation
    )
# Correlate frequency with trajectory class
corr = frequency_trajectory_correlation(freq_df, traj_df)

In [11]:
from src.frequency import (
        save_frequency_results,
    )

In [14]:
query_infinigram("click")

{'ngram': 'click',
 'count': 9132318,
 'approx': False,
 'tokens': ['▁click'],
 'latency_ms': 0.5}